In [27]:
# Phase2_Optim.ipynb
# Author: Marianna Gabrielyan, 20260615
# Email: mgabr001@gmail.com


In [28]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import warnings
warnings.filterwarnings('ignore')
import os

import random
from plotnine  import * 

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestRegressor
import forestci as fci

import dice_ml
from quantile_forest import RandomForestQuantileRegressor
import shap

from sklearn.model_selection import GridSearchCV
# from sklearn.metrics import classification_report

%matplotlib inline


In [29]:
# Read in input file

path = '/home/mgabr001/BrainGenix/BrainEmulationChallenge/src/models/autoassociative/NetmorphParOptim'
os.chdir(path)

df1 = pd.read_excel(open('Phase1_1500_samples-labeled-RS20240628.xlsx','rb'))    
print(df1.shape)
print(df1.columns)

df2 = pd.read_excel(open('Phase0_700_samples-labeled-RS20240628.xlsx','rb'))    
print(df2.shape)
print(df2.columns)

df = pd.concat([df1, df2],  keys=['t1','t2'], axis=0)
df.drop('usable_conns1', axis=1, inplace=True)
df.reset_index(drop=True, inplace=True)

print(df.shape)
print(df.columns)


(1497, 8)
Index(['days', 'pyramidal', 'minneuronseparation', 'shape.radius',
       'shape.thickness', 'dm.weight', 'usable_conns1', 'usable_conns2'],
      dtype='str')
(700, 8)
Index(['days', 'pyramidal', 'minneuronseparation', 'shape.radius',
       'shape.thickness', 'dm.weight', 'usable_conns1', 'usable_conns2'],
      dtype='str')
(2197, 7)
Index(['days', 'pyramidal', 'minneuronseparation', 'shape.radius',
       'shape.thickness', 'dm.weight', 'usable_conns2'],
      dtype='str')


In [30]:
# Read in input file

path = '/home/mgabr001/BrainGenix/BrainEmulationChallenge/src/models/autoassociative/NetmorphParOptim'
os.chdir(path)

# df=pd.read_excel(open('Phase1_1500_samples-labeled-RS20240628.xlsx','rb'))    
df=pd.read_excel(open('Phase0_700_samples-labeled-RS20240628.xlsx','rb'))  
df.drop('usable_conns1', axis=1, inplace=True)
print(df.shape)
print(df.columns)

(700, 7)
Index(['days', 'pyramidal', 'minneuronseparation', 'shape.radius',
       'shape.thickness', 'dm.weight', 'usable_conns2'],
      dtype='str')


In [31]:
df.head()

,days,pyramidal,minneuronseparation,shape.radius,shape.thickness,dm.weight,usable_conns2
0,24,24,14,120,30,0.3,95
1,20,96,15,170,30,0.7,483
2,24,120,15,160,20,0.7,2094
3,20,104,10,130,50,0.5,573
4,25,96,15,110,40,0.9,1769


In [32]:
df1 = df1[(df1['usable_conns2'] <= 2000) & (df1['usable_conns2'] >= 200)]
df1.reset_index(drop=True, inplace=True)
print(df1.shape[0])

df2 = df2[(df2['usable_conns2'] <= 2000) & (df2['usable_conns2'] >= 200)]
df2.reset_index(drop=True, inplace=True)
    
print(df2.shape[0])

1071
498


In [33]:
df = df[(df['usable_conns2'] <= 2500) & (df['usable_conns2'] >= 200)]
df.reset_index(drop=True, inplace=True)
print(df.shape[0])

514


#### Create Training and Test Datasets to make sure there are no additional columns

In [34]:
cols = ['days', 'pyramidal', 'minneuronseparation',
       'shape.radius', 'shape.thickness', 'dm.weight']

In [35]:
X_train, X_test, y_train, y_test = train_test_split(df[cols],df['usable_conns2'], test_size=0.2)
X_train.shape

(411, 6)

In [36]:
X_test.shape

(103, 6)

# Generate Diverse Counterfactual Explanations (DiCE)

### Fine tuning RandomForestQuantileRegressor() hyperparameters.

In [37]:
qrf = RandomForestQuantileRegressor(n_estimators=1000, bootstrap = True, max_samples=None, max_features=4, max_depth=20, min_samples_leaf=2, min_samples_split=5, oob_score = True, default_quantiles=0.025)
qrf.fit(X_train, y_train)

# If your oob_score is high (e.g., > 0.8), model is generalizing well.
# If your oob_score is low but your training score is high, your model is "memorizing" the training simulations 
# and your uncertainty intervals (error bars) will be lies—they will look small, but the model will fail on new unseen simulations.

print("OOB Score = ", qrf.oob_score_)

# Get OOB predictions (Predictions on the training set using only 'unseen' trees)
# This is a 'honest' way to see how the model performs on known data
oob_preds = qrf.predict(X_train[cols], quantiles=[0.025, 0.5, 0.975], oob_score=True)

# Check: Do ~95% of your actual training labels fall within these OOB intervals? 
# If yes, your error bars are well-calibrated.

# Separate the OOB bounds
oob_lower = oob_preds[:, 0]
oob_median = oob_preds[:, 1]
oob_upper = oob_preds[:, 2]


OOB Score =  0.8575342440378725


In [38]:

# 2. Create a "Conservative" Model Wrapper
# This wrapper tells the optimizer to look at the BOTTOM of the 95% interval
# Conservative Strategy: Optimize features until the bottom of the 95% interval is above the threshold. This ensures we hit our goal with 97.5% confidence (one-tailed).
class ConservativeModel:
    def __init__(self, model, threshold):
        self.model = model
        self.threshold = threshold
    
    def predict(self, instances):
        # We predict the 0.025 quantile (Lower Bound)
        # If the Lower Bound > threshold, we are 97.5% sure the real value is too.
        return self.model.predict(instances, quantiles=0.025)

In [39]:

# # Wrap the model for DiCE
# conservative_wrapper = ConservativeModel(qrf, threshold=500)
# m = dice_ml.Model(model=conservative_wrapper, backend="sklearn", model_type='regressor')

# # 3. Setup DiCE with specific NETMORPH features
# # Setup Data with discrete ranges
# # Even if they are integers, we list them in continuous_features 
# # so DiCE can explore the range [min, max]
# d = dice_ml.Data(dataframe=pd.concat([X_train[cols], y_train], axis=1), 
#                  continuous_features=cols, 
#                  outcome_name='usable_conns2')

# exp = dice_ml.Dice(d, m, method="random")


In [40]:

# # Generate Counterfactuals
# # We search for parameters where the LOWER BOUND of the model is > threshold

# query_instance = X_test[cols].iloc[0:2]
# dice_exp = exp.generate_counterfactuals(
#     query_instance, 
#     total_CFs=7, 
#     desired_range=[500.0, 2500.0], # Target range for the lower bound
#     permitted_range={'days': [20,25], 
#                      'pyramidal': [16, 128], 
#                      'minneuronseparation': [10, 15], 
#                      'shape.radius': [100, 200], 
#                      'shape.thickness': [20.0, 50.0], 
#                      'dm.weight': [0.3, 0.7]},
#     features_to_vary=list(cols)
# )

# dice_exp.visualize_as_dataframe(show_only_changes=True)

# cf_df = dice_exp.cf_examples_list[0].final_cfs_df



## Create a function that generates counterfactuals combining the previous cells, and accepts user defined ranges and thresholds.

In [41]:
def generate_counterfactuals(
    query_instance,
    total_CFs,
    desired_range,
    permitted_range,
    features_to_vary,
    threshold=500,
    train_data=X_train,
    train_target=y_train,
):
    # Wrap the quantile RF model so DiCE optimizes the lower bound
    conservative_model = ConservativeModel(qrf, threshold=threshold)
    dice_model = dice_ml.Model(model=conservative_model, backend="sklearn", model_type="regressor")

    dice_data = dice_ml.Data(
        dataframe=pd.concat(
            [train_data[cols].reset_index(drop=True), train_target.reset_index(drop=True)],
            axis=1,
        ),
        continuous_features=cols,
        outcome_name="usable_conns2",
    )

    exp = dice_ml.Dice(dice_data, dice_model, method="random")

    dice_exp = exp.generate_counterfactuals(
        query_instance,
        total_CFs=total_CFs,
        desired_range=desired_range,
        permitted_range=permitted_range,
        features_to_vary=features_to_vary,
    )
    
    cf_df = dice_exp.cf_examples_list[0].final_cfs_df.copy()
    dice_exp.visualize_as_dataframe(show_only_changes=True)
    return dice_exp, cf_df

# Example usage:
# dice_exp, cf_df = generate_counterfactuals(
#     query_instance=X_test[cols].iloc[0:2],
#     total_CFs=10,
#     desired_range=[500.0, 2500.0],
#     permitted_range={
#         'days': [20, 25],
#         'pyramidal': [16, 128],
#         'minneuronseparation': [10, 15],
#         'shape.radius': [100, 200],
#         'shape.thickness': [20.0, 50.0],
#         'dm.weight': [0.3, 0.7],
#     },
#     features_to_vary=cols,
# )

In [42]:
dice_exp, cf_df = generate_counterfactuals(
    query_instance=X_test[cols].iloc[0:2],
    total_CFs=7,
    desired_range=[500.0, 2500.0],
    threshold=500,
    permitted_range={
        'days': [20, 25],
        'pyramidal': [16, 128],
        'minneuronseparation': [10, 15],
        'shape.radius': [100, 200],
        'shape.thickness': [20.0, 50.0],
        'dm.weight': [0.3, 0.7],
    },
    features_to_vary=cols,
)

100%|██████████| 2/2 [00:06<00:00,  3.17s/it]

Query instance (original outcome : 1019.0)


,days,pyramidal,minneuronseparation,shape.radius,shape.thickness,dm.weight,usable_conns2
0,25,96,14,130,40,0.8,1019.0



Diverse Counterfactual set (new outcome: [500.0, 2500.0])


,days,pyramidal,minneuronseparation,shape.radius,shape.thickness,dm.weight,usable_conns2
0,21.0,128.0,-,-,-,-,787.0
1,-,-,-,174.0,28.0,-,567.0
2,-,-,10.0,-,-,0.5,879.0
3,-,-,-,197.0,33.0,-,518.0
4,-,-,-,101.0,22.0,-,831.0
5,-,83.0,-,-,-,-,640.0
6,22.0,-,-,121.0,-,-,567.0


Query instance (original outcome : 879.0)


,days,pyramidal,minneuronseparation,shape.radius,shape.thickness,dm.weight,usable_conns2
0,25,96,10,130,40,0.5,879.0



Diverse Counterfactual set (new outcome: [500.0, 2500.0])


,days,pyramidal,minneuronseparation,shape.radius,shape.thickness,dm.weight,usable_conns2
0,-,-,-,134.0,-,-,-
1,-,110.0,-,-,-,-,1046.0
2,-,93.0,-,-,-,0.4,-
3,-,-,12.0,119.0,-,-,1015.0
4,-,-,-,192.0,35.0,-,501.8500061035156
5,-,-,-,132.0,21.0,-,-
6,-,-,15.0,101.0,-,-,1015.0


#### Predict usable_conns2 on training data using qrf Best Fit Model

Create Upper and lower bounds of 95% Prediction Interval for each prediction in the Training and Test Datasets.

In [ ]:
# # predict on training data
# X_train['Pred_usable_conns2'] = oob_median
# X_train['lower']=oob_lower
# X_train['upper']=oob_upper
# X_train['usable_conns2'] = y_train


In [ ]:
# # Predict on test data. Since the Model has never seen test data, we can use the full model (all trees) to predict on test data (No need for OOB).

# X_test['Pred_usable_conns2'] = qrf.predict(X_test[cols], quantiles=0.5)
# X_test['lower']=qrf.predict(X_test[cols], quantiles=0.025) 
# X_test['upper']=qrf.predict(X_test[cols], quantiles=0.975) 
# X_test['usable_conns2'] = y_test

In [ ]:
# X_train.head()

In [ ]:
# X_test.head()

In [ ]:
# X_train.sort_values(by='usable_conns2', ascending=False, inplace = True)
# X_train.reset_index(drop=True, inplace=True)


# X_test.sort_values(by='usable_conns2', ascending=False, inplace = True)
# X_test.reset_index(drop=True, inplace=True)

In [ ]:
# f, ax = plt.subplots(figsize=(8, 8))

# plt.scatter(X_train['usable_conns2'], X_train['Pred_usable_conns2'], color ='blue', label = 'Training Set')
# plt.scatter(X_test['usable_conns2'],  X_test['Pred_usable_conns2'],color ='r', label = 'Test Set')
# # plt.errorbar(X_test['usable_conns2'],  X_test['Pred_usable_conns2'], yerr = X_test['Pred_Usable_Conns_Err'], fmt='o',color ='r', label = 'Test Set')

# # plt.fill_between(X_train['usable_conns2'], X_train['Pred_usable_conns2']-1.96*X_train['Pred_Usable_Conns_Err'], X_train['Pred_usable_conns2'] + 1.96*X_train['Pred_Usable_Conns_Err'], color='r', alpha=0.25, label = '95% CI')
# # plt.scatter(X_train['usable_conns2'], X_train['Pred_usable_conns2']-1.96*X_train['Pred_Usable_Conns_Err'], color ='cyan', marker='+' )
# # plt.scatter(X_train['usable_conns2'], X_train['Pred_usable_conns2']+1.96*X_train['Pred_Usable_Conns_Err'], color ='green', marker='+' )

# plt.fill_between(X_train['usable_conns2'], X_train['lower'], X_train['upper'], color='r', alpha=0.25, label = '95% PI')
# # plt.scatter(X_train['usable_conns2'], X_train['lower'], color ='cyan', marker='+' )
# # plt.scatter(X_train['usable_conns2'], X_train['upper'], color ='green', marker='+' )

# plt.plot(np.linspace(0,3000,300), np.linspace(0,3000, 300), 'b--', linewidth=3, alpha= 0.2)

# plt.gca().set_facecolor('xkcd:white')
# ax.grid(which='major', color='#DDDDDD', linewidth=0.8)


# ax.spines['bottom'].set_color('k')
# ax.spines['top'].set_color('k')
# ax.spines['left'].set_color('k')
# ax.spines['right'].set_color('k')

# # plt.xlim([0,3500])
# # plt.ylim([0,3500])
# plt.rc('font', size=18) 
# plt.rc('font', size=18) 

# plt.title('Predicted vs Simulated Usable Connections for Training and Test Datasets',size = 20, fontweight="bold")
# plt.xlabel('Simulated Usable Connections (usable_conns2)',size =18)
# plt.ylabel('Predicted Usable Connections (Pred_usable_conns2)',size = 18)
# plt.legend(loc='upper left', prop={'size': 20},framealpha=0.)
# plt.show()
# # f.savefig('Pred_vs_Sim_UsableConns_Train700_Test1500.png')

In [ ]:
# X_train[(X_train['usable_conns2']==0) & (X_train['Pred_usable_conns2'] > 0)]

In [ ]:
# X_train[(X_train['usable_conns2'] > 0) & (X_train['Pred_usable_conns2'] == 0.0)]

In [ ]:
# X_test[(X_test['usable_conns2']==0) & (X_test['Pred_usable_conns2'] > 0)]

In [ ]:
# X_test[(X_test['usable_conns2']==0) & (X_test['Pred_usable_conns2'] > 0)]